## Prueba de las transformaciones de los datos 

In [1]:
import pandas as pd
schedule = pd.read_csv(r"C:\Users\poloc\Documents\PruebasGit\Primera prueba\data\raw\staff_schedule.csv")
patients = pd.read_csv(r"C:\Users\poloc\Documents\PruebasGit\Primera prueba\data\raw\patients.csv")
staff = pd.read_csv(r"C:\Users\poloc\Documents\PruebasGit\Primera prueba\data\raw\staff.csv")
service = pd.read_csv(r"C:\Users\poloc\Documents\PruebasGit\Primera prueba\data\raw\services_weekly.csv")

In [2]:
staff.head()

,staff_id,staff_name,role,service
0,STF-5ca26577,Allison Hill,doctor,emergency
1,STF-02ae59ca,Noah Rhodes,doctor,emergency
2,STF-d8006e7c,Angie Henderson,doctor,emergency
3,STF-212d8b31,Daniel Wagner,doctor,emergency
4,STF-107a58e4,Cristian Santos,doctor,emergency


### Transformaciones a los datos
Por tipos de datos
1. Quitar el -PAT de patients_id en "Patients"
2. Convertir a tipo Date los datos de "arrivale_date" y "deperture_date" de Patients
3. Quitar el -STF de "staff_id" de Staff

Transformaciones del modelo
1. Crear una nueva tabla llamada "Departaments" que tiene dos columnas "service_id"(TEXT) y generarla aleatoriamente. Otra columna de service_name(TEXT) y que tiene los servicios del hospital de manera única
2. Generar una nueva tabla llamada "Ingreso_paciente" que viene de la tabla patients y que tiene lo siguiente: Un ingreso_id(text) generado de manera random en la creación de la tabla, un patient_id, service_id, arrivale_date, depature_date y satisfaction
3. Dejar la tabla de patients solamente con: patient_id, dividir los nombres y apellidos así: name, y last_name, y una nueva columna de bith_date que sea restar el número de años del dato age (que sea tipo DATE)
4. Cambiar el nombre de la tabla de service a "operational_shift" y va a tener las siguientes columnas: un resgistro_id generado de manera random. un service_id en vez de service, week, borrar el month, available_beds, patients_request, patients_refused, patients_satisfaction, staff_morale y event
5. Utilizar la tabla de schudele como "Staff_attendence" con asistencia_id, week, staff:id, departament_id y present. 
6. Dejar staff solamente con Staff_id, staff_name, staff_last_name, role

Cada cosa es una función

In [3]:
# Quitar el -PAT de patients_id en "Patients"
patients['patient_id'] = patients['patient_id'].str.replace('PAT-', '', regex=False)
patients['arrival_date'] = pd.to_datetime(patients['arrival_date'])
patients['departure_date'] = pd.to_datetime(patients['departure_date'])
staff['staff_id'] = staff['staff_id'].str.replace('STAF-', '')

In [4]:
patients.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction
0,09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61
1,f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83
2,ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83
3,3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81
4,08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76


Transformación del modelo (voy a borrar todo al final)

In [5]:
# Crear una nueva tabla llamada "Departaments" que tiene dos columnas "service_id"(TEXT) y generarla aleatoriamente. Otra columna de service_name(TEXT) y que tiene los servicios del hospital de manera única
departaments = pd.DataFrame({
    "departament": staff["service"].unique()  
})
departaments.insert(0, 'service_id', range(1, len(departaments) + 1))
departaments.head()

,service_id,departament
0,1,emergency
1,2,surgery
2,3,general_medicine
3,4,ICU


In [6]:
# Generar una nueva tabla llamada "Ingreso_paciente" que viene de la tabla patients y que tiene lo siguiente: Un ingreso_id(text) generado de manera random en la creación de la tabla, un patient_id, service_id, arrivale_date, depature_date y satisfaction
ingreso_paciente = patients.copy()
ingreso_paciente.drop(columns=["name","age"], inplace=True)
ingreso_paciente.insert(0, 'ingreso_id', range(1, len(ingreso_paciente) + 1))
ingreso_paciente.head()


,ingreso_id,patient_id,arrival_date,departure_date,service,satisfaction
0,1,09484753,2025-03-16,2025-03-22,surgery,61
1,2,f0644084,2025-12-13,2025-12-14,surgery,83
2,3,ac6162e4,2025-06-29,2025-07-05,general_medicine,83
3,4,3dda2bb5,2025-10-12,2025-10-23,emergency,81
4,5,08591375,2025-02-18,2025-02-25,ICU,76


In [7]:
maps_id = departaments.set_index("departament")["service_id"]
ingreso_paciente["service"] = ingreso_paciente["service"].map(maps_id)

In [8]:
ingreso_paciente.rename(columns={"service":"service_id"}, inplace=True)
ingreso_paciente.head()

,ingreso_id,patient_id,arrival_date,departure_date,service_id,satisfaction
0,1,09484753,2025-03-16,2025-03-22,2,61
1,2,f0644084,2025-12-13,2025-12-14,2,83
2,3,ac6162e4,2025-06-29,2025-07-05,3,83
3,4,3dda2bb5,2025-10-12,2025-10-23,1,81
4,5,08591375,2025-02-18,2025-02-25,4,76


In [9]:
# Dejar la tabla de patients solamente con: patient_id, dividir los nombres y apellidos así: name, y last_name, y una nueva columna de bith_date que sea restar el número de años del dato age (que sea tipo DATE)
patients[["name_patient", "last_name_patient"]] = patients["name"].str.split(" ", n=1, expand=True)
patients.drop(columns=["name"], inplace=True)
patients.drop(columns=["arrival_date"], inplace=True)
patients.drop(columns=["departure_date"], inplace=True)
patients.drop(columns=["service"], inplace=True)
patients.drop(columns=["satisfaction"], inplace=True)
mes_fijado = "-04-10"
nacimiento = (2025 - patients["age"]).astype(str) + mes_fijado
patients["birth_date"] = pd.to_datetime(nacimiento)
patients.drop(columns=["age"], inplace=True)
patients.head()

,patient_id,name_patient,last_name_patient,birth_date
0,09484753,Richard,Rodriguez,2001-04-10
1,f0644084,Shannon,Walker,2019-04-10
2,ac6162e4,Julia,Torres,2001-04-10
3,3dda2bb5,Crystal,Johnson,1993-04-10
4,08591375,Garrett,Lin,2000-04-10


In [10]:
# Cambiar el nombre de la tabla de service a "operational_shift" y va a tener las siguientes columnas: un resgistro_id generado de manera random. un service_id en vez de service, week, borrar el month, available_beds, patients_request, patients_refused, patients_satisfaction, staff_morale y event
operational_shift = service.copy()
operational_shift.insert(0, 'shift_id', range(1, len(operational_shift) + 1))
maps_id = departaments.set_index("departament")["service_id"]
operational_shift["service"] = operational_shift["service"].map(maps_id)
operational_shift.rename(columns={"service":"service_id"}, inplace=True)

In [11]:
operational_shift.drop(columns=["month"], inplace=True)
operational_shift.head()

,shift_id,week,service_id,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event
0,1,1,1,32,76,32,44,67,70,none
1,2,1,2,45,130,45,85,83,78,flu
2,3,1,3,37,201,37,164,97,43,flu
3,4,1,4,22,31,22,9,84,91,flu
4,5,2,1,28,169,28,141,75,64,none


In [12]:
# 5. Utilizar la tabla de schudele como "Staff_attendence" con asistencia_id, week, staff:id, departament_id y present. Dejar staff solamente con Staff_id, staff_name, staff_last_name, role
staff_attendance = schedule.copy()
staff_attendance['staff_id'] = staff_attendance['staff_id'].str.replace('STF-', '', regex=False)
maps_id = departaments.set_index("departament")["service_id"]
staff_attendance["service"] = staff_attendance["service"].map(maps_id)
staff_attendance.rename(columns={"service":"service_id"}, inplace=True)
staff_attendance.drop(columns=["staff_name", "role"], inplace=True)
staff_attendance.head()

,week,staff_id,service_id,present
0,1,b77cdc60,1,1
1,2,b77cdc60,1,1
2,3,b77cdc60,1,0
3,4,b77cdc60,1,1
4,5,b77cdc60,1,1


In [13]:
staff_attendance.insert(0, 'attendance_id', range(1, len(staff_attendance) + 1))

In [14]:

staff_attendance.head()

,attendance_id,week,staff_id,service_id,present
0,1,1,b77cdc60,1,1
1,2,2,b77cdc60,1,1
2,3,3,b77cdc60,1,0
3,4,4,b77cdc60,1,1
4,5,5,b77cdc60,1,1


In [15]:

staff['staff_id'] = staff['staff_id'].str.replace('STF-', '', regex=False)
staff.head()

,staff_id,staff_name,role,service
0,5ca26577,Allison Hill,doctor,emergency
1,02ae59ca,Noah Rhodes,doctor,emergency
2,d8006e7c,Angie Henderson,doctor,emergency
3,212d8b31,Daniel Wagner,doctor,emergency
4,107a58e4,Cristian Santos,doctor,emergency


In [16]:
staff[["name_staff", "last_name_staff"]] = staff["staff_name"].str.split(" ", n=1, expand=True)
staff.drop(columns=["staff_name"], inplace=True)

In [17]:
staff.drop(columns=["service"], inplace=True)
staff.head()

,staff_id,role,name_staff,last_name_staff
0,5ca26577,doctor,Allison,Hill
1,02ae59ca,doctor,Noah,Rhodes
2,d8006e7c,doctor,Angie,Henderson
3,212d8b31,doctor,Daniel,Wagner
4,107a58e4,doctor,Cristian,Santos


Todas las tablas que vamos a usar

In [18]:
staff.head()


,staff_id,role,name_staff,last_name_staff
0,5ca26577,doctor,Allison,Hill
1,02ae59ca,doctor,Noah,Rhodes
2,d8006e7c,doctor,Angie,Henderson
3,212d8b31,doctor,Daniel,Wagner
4,107a58e4,doctor,Cristian,Santos


In [19]:
staff_attendance.head()


,attendance_id,week,staff_id,service_id,present
0,1,1,b77cdc60,1,1
1,2,2,b77cdc60,1,1
2,3,3,b77cdc60,1,0
3,4,4,b77cdc60,1,1
4,5,5,b77cdc60,1,1


In [20]:
operational_shift.head()


,shift_id,week,service_id,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event
0,1,1,1,32,76,32,44,67,70,none
1,2,1,2,45,130,45,85,83,78,flu
2,3,1,3,37,201,37,164,97,43,flu
3,4,1,4,22,31,22,9,84,91,flu
4,5,2,1,28,169,28,141,75,64,none


In [21]:
ingreso_paciente.head()

,ingreso_id,patient_id,arrival_date,departure_date,service_id,satisfaction
0,1,09484753,2025-03-16,2025-03-22,2,61
1,2,f0644084,2025-12-13,2025-12-14,2,83
2,3,ac6162e4,2025-06-29,2025-07-05,3,83
3,4,3dda2bb5,2025-10-12,2025-10-23,1,81
4,5,08591375,2025-02-18,2025-02-25,4,76


In [22]:
departaments.head()

,service_id,departament
0,1,emergency
1,2,surgery
2,3,general_medicine
3,4,ICU


In [23]:

patients.head()

,patient_id,name_patient,last_name_patient,birth_date
0,09484753,Richard,Rodriguez,2001-04-10
1,f0644084,Shannon,Walker,2019-04-10
2,ac6162e4,Julia,Torres,2001-04-10
3,3dda2bb5,Crystal,Johnson,1993-04-10
4,08591375,Garrett,Lin,2000-04-10
